# CVD_내부 합성변화 실험 — 주입 크기·위치·방향을 바꾼 Test 평가

입력·실행·핵심 결과 순서로 읽습니다. 기본 화면은 요약표이며 `SHOW_DETAILS=True`일 때만 세부 표를 표시합니다. 기존 모델은 변경하지 않습니다.

PECVD 원본 Test를 사용하는 내부 연구입니다. 외부 제공 주입 CSV를 읽는 시험은 notebooks/CVD_predict_external_csv.ipynb입니다.


In [ ]:
from pathlib import Path
import sys, importlib, os
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'src'/'pecvd').is_dir())
if str(ROOT/'src') not in sys.path: sys.path.insert(0,str(ROOT/'src'))
import pecvd.experiments.pipeline_settings as pipeline_settings
pipeline_settings=importlib.reload(pipeline_settings)
PIPELINE_SETTINGS=pipeline_settings.configure_pipeline(ROOT)
if PIPELINE_SETTINGS.get('decision_system')!='fusion_answers':
    raise RuntimeError('These experiments require the trained fusion_answers model.')
from pecvd.experiments.questions import PRIMARY_GOAL_VERSION
from pecvd.experiments.fusion_answers import SYSTEM as GEVIS_SYSTEM
print({'model':GEVIS_SYSTEM,'training_profile':PIPELINE_SETTINGS['training_profile'],'weights':'existing STEP1/2/3'})


## 1. 학습된 Fast GEVIS 로딩과 실험 설정

STEP 3가 완료됐다면 각 EXP의 커널을 재시작하고 처음부터 실행합니다. 원본 CSV를 읽거나 가중치를 다운로드하지 않습니다.
결과는 기밀이며 자동 파일 저장·업로드는 하지 않습니다.


In [ ]:
from dataclasses import replace
import pandas as pd
import pecvd.experiments.routing_audit as runtime_refresh
runtime_refresh=importlib.reload(runtime_refresh)
runtime_refresh.refresh_evaluation_modules()
for _module in ('fusion_answers','fusion_metrics','fast_experiments'):
    importlib.reload(importlib.import_module('pecvd.experiments.'+_module))
from pecvd.experiments import fusion_answers as fusion_answers, fast_experiments as fast
RUNTIME=fast.load_runtime(ROOT)
SEED=42
SELECTED_STEPS=None
MAX_WINDOWS_PER_STEP=16
CONFIG=replace(fusion_answers.FusionConfig(),seed=SEED,max_windows_per_step=MAX_WINDOWS_PER_STEP)
print({'decision':'question-conditioned Fusion + trained selected-center head',
       'question_scope':'fixed English questions; sensor selection is separate',
       'answer_head_parameters':sum(p.numel() for p in RUNTIME.module.FUSION_ANSWER_HEAD.parameters()),
       'deployed_weights_updated':False})
CHALLENGES=fast.CHALLENGES

SHOW_DETAILS=False


## 2. 실제 Fast 모델 실험 실행

모델이 없거나 설정이 맞지 않으면 중단합니다. 데이터나 모델을 모의 결과로 대체하지 않습니다.


In [ ]:
REPORT=fast.run_generalization(RUNTIME,steps=SELECTED_STEPS,config=CONFIG,challenges=CHALLENGES)
SUMMARY=REPORT.summary;DETAILS=REPORT.details
display(SUMMARY[[c for c in ('variant','answer_type','questions','answer_accuracy','macro_f1') if c in SUMMARY]])


## 조건별 클래스 성능과 미평가 답변

전체 평균만 보지 말고 새 변화에서 `Yes` recall과 변화 유형별 F1이 얼마나 떨어지는지 확인합니다.
ORDINAL은 물리적 심각도가 아니라 지정한 합성 변화 강도의 단계입니다.


In [ ]:
if SHOW_DETAILS:
    for _name,_report in REPORT.reports.items():
        print(_name,'per-class metrics')
        display(_report.diagnostics['per_class'])
        _unscored=_report.diagnostics['unscored_answers']
        print('Unscored answers:',len(_unscored))
    ERROR_CASES=DETAILS.loc[~DETAILS.correct].copy()
    display(ERROR_CASES[['variant','step','answer_type','center_sensor','label','prediction']].head(20))
